# Preprocessing
## Etapes à faire suite à l'EDA, avant de commencer le modèle linéaire
- Eliminer quelque valeurs extrème de `blood pressure`
- Transformer la distribution (right skewed) de la target en distribution normale
- Attention aux multicolinearité (VIF>=10) bon jsp pourquoi mais on voit dans l'EDA qu'elles le sont toutes, mais je vais devoir check si je fais bien le calcule et on pourra le notifier dans le rapport 
- BMI au lieu de weight-height (feature engeneering)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, OrdinalEncoder

import joblib

## 1. Chargement des données

In [ ]:
X_train = pd.read_csv("../data/data_labeled/X_train.csv")
y_train = pd.read_csv('../data/data_labeled/y_train.csv').squeeze()
X_test = pd.read_csv('../data/data_labeled/X_test.csv')
y_test = pd.read_csv('../data/data_labeled/y_test.csv').squeeze()

print(f"Shape X_train avant preprocessing : {X_train.shape}")
print(f"Shape y_train avant preporcessing : {y_train.shape}")

## 2. Retirer les outliers de blood pressure

In [ ]:
plt.figure(figsize=(10, 4))
sns.boxplot(y=X_train['blood pressure'])
plt.title('Blood pressure - AVANT')
plt.show()

threshold_bloodpressure = 150
X_train_clean = X_train[X_train['blood pressure'] < threshold_bloodpressure].copy()
y_train_clean = y_train[X_train['blood pressure'] < threshold_bloodpressure].copy()

plt.figure(figsize=(10, 4))
sns.boxplot(y=X_train_clean["blood pressure"])
plt.title(f"Blood pressure - APRES - threshold = {threshold_bloodpressure}")
plt.show()

# a decommenter/recommenter plus tard si jamais on a un pb aux resultats
X_test_clean = X_test[X_test['blood pressure'] < threshold_bloodpressure].copy()
y_test_clean = y_test[X_test['blood pressure'] < threshold_bloodpressure].copy()

### Feature engeneering : IMC au lieu de height & weight !


In [ ]:
def add_bmi(X):
    """Ajoute BMI = weight / (height/100)^2"""
    X = X.copy()
    X['bmi'] = X['weight'] / (X['height'] / 100) ** 2
    return X

X_train_fe = add_bmi(X_train_clean)
X_test_fe = add_bmi(X_test_clean)

print(f"\nFeatures après engineering : {X_train_fe.columns.tolist()}")

## 3. Transformation de la target

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.hist(y_train_clean, bins=50, edgecolor='black')
plt.title('Target - ORIGINALE')
plt.xlabel('Risk')

# Tester log transform
y_train_log = np.log1p(y_train_clean)  # log1p = log(1+x) pour éviter log(0)
plt.subplot(1, 3, 2)
plt.hist(y_train_log, bins=50, edgecolor='black')
plt.title('Target - LOG TRANSFORM')
plt.xlabel('log(1 + Risk)')

# Tester sqrt transform
y_train_sqrt = np.sqrt(y_train_clean)
plt.subplot(1, 3, 3)
plt.hist(y_train_sqrt, bins=50, edgecolor='black')
plt.title('Target - SQRT TRANSFORM')
plt.xlabel('sqrt(Risk)')

plt.tight_layout()
plt.show()

# checker la ressemblence avec la loi normale normalité
from scipy.stats import normaltest
stat_original, p_original = normaltest(y_train_clean)
stat_log, p_log = normaltest(y_train_log)
stat_sqrt, p_sqrt = normaltest(y_train_sqrt)

print("\nTests de normalité (p-value > 0.05 = normal)")
print(f"Original : p={p_original:.10f}")
print(f"Log      : p={p_log:.10f}")
print(f"Sqrt     : p={p_sqrt:.10f}")

# Choisir la meilleure transformation
# (Celle avec la p-value la plus haute)
if p_log > p_sqrt and p_log > p_original:
    print("\nOn choisit LOG TRANSFORM")
    y_train_transformed = y_train_log
    y_test_transformed = np.log1p(y_test_clean)
    transform_type = 'log'
elif p_sqrt > p_original:
    print("\nOn choisit SQRT TRANSFORM")
    y_train_transformed = y_train_sqrt
    y_test_transformed = np.sqrt(y_test_clean)
    transform_type = 'sqrt'
else:
    print("\nOn garde ORIGINAL")
    y_train_transformed = y_train_clean
    y_test_transformed = y_test_clean
    transform_type = 'none'


### Commentaires sur les résultats

on va juste tester les 3 types (sans tranfo - log - sqrt) et voir s'il y a une différence de rmse significative

## 4. Mapping des variables (ordinales)
### Variables autre que `profession`

In [ ]:
ordinal_features = ['sarsaparilla', 'smurfberry liquor', 'smurfin donuts']
ordinal_order = [["Very high", "High", "Moderate", "Low", "Very low"]]*3
print(ordinal_order)
encoder = OrdinalEncoder(categories = ordinal_order)

X_train_encoded = X_train.copy()
X_train_encoded[ordinal_features] = encoder.fit_transform(
    X_train[ordinal_features]
)

X_test_encoded = X_test.copy()
X_test_encoded[ordinal_features] = encoder.fit_transform(
    X_test[ordinal_features]
)

print(X_train_encoded[ordinal_features].head(6+1))
print(X_test_encoded[ordinal_features].head())

print(X_train_encoded[ordinal_features].isnull().sum())

### Encodage de `profession`
On sait qu'il n'y a pas trop de relation entre les différentes professions via EDA -> one hot encoding

In [ ]:
print("Liste des features AVANT")
X_train_encoded.info()

X_train_encoded = pd.get_dummies(
    X_train_encoded,
    columns = ['profession'],
    drop_first = True # eviter la multicolinearité
)
X_test_encoded = pd.get_dummies(
    X_test_encoded, 
    columns=['profession'], 
    drop_first=True
)

# (au cas où une profession n'apparaît que dans train ou test)
# Ajouter une colonnes manquantes dans test
missing_cols = set(X_train_encoded.columns) - set(X_test_encoded.columns)
for col in missing_cols:
    X_test_encoded[col] = 0

# Supprimer colonnes en trop dans test
extra_cols = set(X_test_encoded.columns) - set(X_train_encoded.columns)
X_test_encoded = X_test_encoded.drop(columns=list(extra_cols))

# réordonner pour avoir le même ordre
X_test_encoded = X_test_encoded[X_train_encoded.columns]

print(f"Nombre de colonnes : {X_train_encoded.shape[1]}")
print(f"Colonnes train = colonnes test : {list(X_train_encoded.columns) == list(X_test_encoded.columns)}")

print("########################")
print("Liste des features APRES")
X_train_encoded.info()

## 5. Data transformation

### Normalisation et rescaling


In [ ]:
# supp image_path avant la standardisation

if "img_filename" in X_train_encoded.columns:
    img_filename_train = X_train_encoded["img_filename"]
    img_filename_test  = X_test_encoded["img_filename"]
    X_train_encoded    = X_train_encoded.drop(columns = ["img_filename"])
    X_test_encoded     = X_test_encoded.drop(columns = ["img_filename"])

# scale sur train uniquement!!
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded) # transform pas fit !!

X_train_final = pd.DataFrame(X_train_scaled, columns=X_train_encoded.columns)
X_test_final = pd.DataFrame(X_test_scaled, columns=X_test_encoded.columns)

print(f"X_train final shape : {X_train_final.shape}")
print(f"X_test final shape  : {X_test_final.shape}")
X_train_final.describe()

## Fin - Sauvegarde

In [ ]:
joblib.dump(scaler, '../results/models/scaler.pkl')
joblib.dump(ordinal_order, '../results/models/ordinal_order.pkl')
joblib.dump(X_train_encoded.columns.tolist(), '../results/models/feature_names.pkl')

print("\nScaler et paramètres sauvegardés dans results/models/")